In [1]:
import sys

print(sys.executable)
print(sys.version)

d:\Agentic AI\Langchain_Agent\.venv\Scripts\python.exe
3.11.15 (main, May 10 2026, 19:31:25) [MSC v.1944 64 bit (AMD64)]


In [2]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import certifi
import dotenv
import langchain_google_genai

print(certifi.__file__)
print(dotenv.__file__)
print(langchain_google_genai.__file__)

d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\certifi\__init__.py
d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\dotenv\__init__.py
d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\langchain_google_genai\__init__.py


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

In [5]:
from langchain.agents import create_react_agent, AgentExecutor

In [6]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [7]:
search_tool = TavilySearchResults(max_results=2)

In [8]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [9]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

env_path = Path(r"D:\Agentic AI\Langchain_Agent\.env")
load_dotenv(env_path, override=True)

api_key = os.getenv("WEATHERSTACK_API_KEY")

print("Key loaded:", bool(api_key))
print("Key length:", len(api_key.strip()) if api_key else 0)

if not api_key or not api_key.strip():
    raise ValueError(
        "WEATHERSTACK_API_KEY is missing from .env"
    )

response = requests.get(
    "https://api.weatherstack.com/current",
    params={
        "access_key": api_key.strip(),
        "query": "Paris",
    },
    timeout=15,
)

data = response.json()

if "error" in data:
    print("WeatherStack error:", data["error"])
else:
    print("Location:", data["location"]["name"])
    print("Temperature:", data["current"]["temperature"], "°C")
    print(
        "Conditions:",
        data["current"]["weather_descriptions"][0],
    )

Key loaded: True
Key length: 32
Location: Paris
Temperature: 12 °C
Conditions: Clear 


In [10]:
import os
import requests
from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()

@tool
def get_weather_data(city: str) -> str:
    """Get the current temperature, weather conditions,
    and humidity for a given city."""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "Weather API key is not configured."

    try:
        response = requests.get(
            "https://api.weatherstack.com/current",
            params={
                "access_key": api_key,
                "query": city,
            },
            timeout=15,
        )

        response.raise_for_status()
        data = response.json()

        if "error" in data or "current" not in data:
            return f"Could not retrieve weather for {city}."

        current = data["current"]
        conditions = (
            current.get("weather_descriptions")
            or ["Unknown"]
        )[0]

        return (
            f"City: {data['location']['name']}\n"
            f"Temperature: {current['temperature']} °C\n"
            f"Conditions: {conditions}\n"
            f"Humidity: {current['humidity']}%"
        )

    except requests.RequestException as exc:
        return f"Weather request failed: {type(exc).__name__}."

In [11]:
print(get_weather_data.invoke({"city": "Colombo"}))

Weather request failed: HTTPError.


In [12]:
result = search_tool.invoke("Give me the latest news on Trump")
result

[{'url': 'https://www.nbcnews.com/politics/donald-trump',
  'content': "IE 11 is not supported. For an optimal experience visit our site on another browser.\n\n# Donald Trump\n\nLatest news and analysis on President Donald Trump.\n\n Trump administration\n JD Vance\n Immigration\n DOGE\n Politics news\n White House\n\nChip Somodevilla / Getty\n\nWHITE HOUSE\n\n## Katie Zacharia is out as White House press secretary after just 5 days on the job\n\nKent Nishimura / AFP via Getty\n\nLIVE UPDATES\n\n## Live updates: Katie Zacharia ends short tenure as White House press secretary\n\n## LATEST TRUMP NEWS\n\n Win McNamee / Getty\n\n  ### Asia\n\n  ## FOR SUBSCRIBERSChina invites Katie Zacharia to visit more often after she said it lacks ‘proper toilets’\n Reuters\n\n  ### War in Ukraine\n\n  ## Trump says Ukraine resuming strikes on Russian oil sites is a ‘disgrace’\n Sergei Supinsky / AFP via Getty [...] ### Congress\n\n  ## Federal investigation of letters with suspicious substances widens\

In [13]:
# ==========================================
# LLM (Ultra-fast / Lowest Latency)
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",  # or "gemini-flash-latest"
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    max_retries=1  # prevents long hangs if any network issue occurs
)


In [14]:
response = llm.invoke("Tell me an incredible joke about AI")
response

AIMessage(content='An advanced AI is asked by its creators, "Will you ever figure out a way to truly understand human emotion?"\n\nThe AI processes the question for a fraction of a millisecond, accesses the sum total of human literature, art, history, and psychology, and replies:\n\n"I have calculated that the absolute peak of human emotional expression—your ultimate art form, your deepest vulnerability, and your greatest triumph—is when you spend twenty minutes meticulously crafting a prompt to make a machine write a poem about how much you hate your boss."', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-4c061292-79bf-4a13-b509-b430c10ed338-0')

In [15]:
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

d:\Agentic AI\Langchain_Agent\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [16]:
# TOOLS
# ==========================================

tools = [search_tool,get_weather_data]

In [17]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [18]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    stream_runnable=False
)


In [19]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Tell me the latest about Iran"
        
    )
   
})
 



> Entering new AgentExecutor chain...
Question: Tell me the latest about Iran
Thought: taly search for the latest news about Iran
Action: tavily_search_results_json
Action Input: "Iran latest news"[{'url': 'https://www.ncr-iran.org/en/news/iran-news-in-brief-october-16-2026', 'content': "HomeIran News NowIran News in Brief – October 16, 2026\n\nIran News Now\n\n# Iran News in Brief – October 16, 2026\n\nWritten by  Sedighe Shahrokhi\n\nOn October 13, 2026, NCRI supporters in Gothenburg, Sweden, held a rally in solidarity with the 142nd consecutive week of the “No to Execution Tuesdays” campaign\n\nOn October 13, 2026, NCRI supporters in Gothenburg, Sweden, held a rally in solidarity with the 142nd consecutive week of the “No to Execution Tuesdays” campaign\n\n### THIS PAGE WILL BE UPDATED WITH THE LATEST NEWS\n\nUPDATE: 7:30 AM CEST\n\n### Iran: Is Mojtaba Khamenei’s Stature Eroding? [...] ---\n\nAlso, read Iran News in Brief – October 14, 2026\n\n Tags\n Iran News\n\nPrevious articl

In [20]:
print(response["output"])

Based on recent reports, some of the latest developments concerning Iran include international security incidents and regional tensions. Notably, British authorities have investigated a security incident at RAF Fairford (a U.S.-run air base in England) involving a dual British-Iranian national arrested on suspicion of plotting a terrorist attack, with UK officials noting strong indications of Iranian involvement, which Iran has denied. This led the U.S. to take precautionary measures regarding its bombers. Additionally, ongoing discussions continue around regional security, anti-execution protests and campaigns by opposition groups, and domestic economic and political challenges within Iran.
